# Notebook 01c — Feature Engineering & TCI Merge

## README

### Overview

This notebook takes the wide-format panel CSVs produced by `01b` and:

1. Merges the **static spatial features** produced by `01b5` (population, building height, road length, public transport, POI counts, power plants) onto every row via a **left join on `spatial_id`** — the same per-tile values broadcast across all dates for that tile. Missing static features for a city (e.g. `01b5` hasn't been run yet) produce a warning and all-NaN static columns rather than blocking the city, consistent with `01b5`'s own missing-input handling.
2. Optionally creates a **1-day lagged NO₂ feature** (`no2_lag1`) per `(city, spatial_id)` group. Rows that cannot have a lag (first day of each city's date range) are dropped.
3. Merges the **Traffic Congestion Index (TCI)** from `processed/{city}/tci/{city}_tci_2025.csv` into the panel. Rows with no matching TCI are dropped (inner join on `spatial_id` × `date`).
4. Reports **missingness** introduced at each step so the user understands data loss.

### Inputs (per city)

| Source | Path | Notes |
|---|---|---|
| `01b_raster_to_grid_aggregation.ipynb` | `data/processed/{city}/grid_panel/{city}_grid_panel.csv.gz` | Wide panel (no2, era5_temp, …) |
| `01b5_spatial_features.ipynb` | `data/processed/{city}/grid_panel/{city}_static_features.csv` | Static per-tile features (no `date` column); optional — missing file ⇒ NaN columns, not a hard failure |
| World Bank | `data/processed/{city}/tci/{city}_tci_2025.csv` | Columns: `date`, `spatial_id`, `length` (→ renamed `tci`) |

### Output (per city)

```
data/processed/{city}/grid_panel/
  {city}_modelling_panel.csv.gz    ← panel + static features + tci [+ no2_lag1]
```

### Output schema

```
city | spatial_id | date | no2 | era5_temp |
tile_pop | building_height | road_length_*_m | road_length_total_m |
n_bus_stops | n_mass_transit_lines | n_mass_transit_polygons | n_public_transport_total |
n_commercial | n_offices | n_education | n_leisure | n_power_plant |
[no2_lag1] | tci
```

### Pipeline position

```
01b_raster_to_grid_aggregation.ipynb
01b5_spatial_features.ipynb
  └─► 01c_feature_engineering.ipynb  ◄── YOU ARE HERE
        └─► 01d_monthly_tile_panel.ipynb
```

### Design principles

- **City-agnostic**: add cities by appending to `CITIES`.
- **Transparent data loss**: every step that drops rows reports counts and percentages.
- **Static features never drop rows**: the `01b5` merge is a left join on `spatial_id` only — it broadcasts the same static value across every date for a tile and never removes a row. A missing `01b5` output degrades to all-NaN static columns with a printed warning, the same convention `01b5` itself uses for missing raw inputs.
- **Lag is optional**: controlled by `ADD_NO2_LAG` flag; set to `False` to skip entirely.
- **Inner join on TCI**: rows without a matching TCI observation are dropped — the dependent variable must be present for modelling.


## Part 0 — Environment Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries loaded.")

## Part 1 — USER INPUTS

**Only edit this cell** to configure cities and feature options.

In [ ]:
# =============================================================================
# ██████████████████████   USER INPUTS   ██████████████████████
# =============================================================================

# --- Project root and data directory ---
PROJECT_ROOT = Path().resolve().parent
DATA_ROOT    = PROJECT_ROOT / "data"

# --- Cities to process ---
CITIES = [
    # {"name": "abuja",        "display_name": "Abuja"},
    # {"name": "algiers",      "display_name": "Algiers"},
    {"name": "baghdad",      "display_name": "Baghdad"},
    {"name": "buenos_aires", "display_name": "Buenos Aires"},
    # {"name": "cairo",        "display_name": "Cairo"},
    # {"name": "cape_town",    "display_name": "Cape Town"},
    # {"name": "dakar",        "display_name": "Dakar"},
    # {"name": "lima",         "display_name": "Lima"},
    # {"name": "los_angeles",  "display_name": "Los Angeles"},
    # {"name": "madrid",       "display_name": "Madrid"},
    {"name": "mexico_city",  "display_name": "Mexico City"},
    # {"name": "mumbai",       "display_name": "Mumbai"},
    {"name": "new_york",     "display_name": "New York"},
    # {"name": "santiago",     "display_name": "Santiago"},
    # {"name": "yangon",       "display_name": "Yangon"},
]

# --- Feature options ---
ADD_NO2_LAG = False   # True → create no2_lag1 column (1-day lag per spatial_id)
                     # Rows that cannot have a lag (first date per city) are dropped.

# --- TCI source column name (will be renamed to 'tci') ---
TCI_SOURCE_COL = "length"

# =============================================================================
# END OF USER INPUTS
# =============================================================================

print(f"Cities    : {[c['name'] for c in CITIES]}")
print(f"NO2 lag   : {ADD_NO2_LAG}")

## Part 2 — Helper Functions

In [ ]:
def load_panel(city_name: str) -> pd.DataFrame:
    """Load the 01b grid panel CSV (gzip or plain) for a city."""
    panel_dir = DATA_ROOT / "processed" / city_name / "grid_panel"

    # Accept both compressed and plain CSV
    for fname in [f"{city_name}_grid_panel.csv.gz", f"{city_name}_grid_panel.csv"]:
        p = panel_dir / fname
        if p.exists():
            df = pd.read_csv(p, parse_dates=["date"])
            print(f"  Loaded panel : {fname}  ({len(df):,} rows)")
            return df

    raise FileNotFoundError(f"No panel CSV found in {panel_dir}")


def load_static_features(city_name: str) -> pd.DataFrame | None:
    """
    Load the 01b5 static spatial-features CSV for a city, if it exists.

    Returns None (not an exception) if the file is missing — the merge step
    handles this by producing all-NaN static columns and printing a warning,
    consistent with 01b5's own graceful-degradation convention for missing
    raw inputs. This keeps a not-yet-run 01b5 from blocking the whole pipeline.
    """
    static_path = DATA_ROOT / "processed" / city_name / "grid_panel" / f"{city_name}_static_features.csv"
    if not static_path.exists():
        print(f"  ⚠️  Static features not found: {static_path}")
        print(f"      (run 01b5_spatial_features.ipynb for this city — proceeding with NaN static columns)")
        return None

    static = pd.read_csv(static_path)
    if "spatial_id" not in static.columns:
        print(f"  ⚠️  'spatial_id' column missing in {static_path.name} — proceeding with NaN static columns")
        return None

    print(f"  Loaded static: {static_path.name}  ({len(static):,} tiles, "
          f"{len(static.columns) - 1} feature columns)")
    return static


def load_tci(city_name: str) -> pd.DataFrame:
    """Load the TCI CSV for a city and rename 'length' → 'tci'."""
    tci_path = DATA_ROOT / "processed" / city_name / "tci" / f"{city_name}_tci_2025.csv"
    if not tci_path.exists():
        raise FileNotFoundError(f"TCI file not found: {tci_path}")

    tci = pd.read_csv(tci_path, parse_dates=["date"])
    tci = tci.rename(columns={TCI_SOURCE_COL: "tci"})

    # Keep only the columns needed for the merge
    tci = tci[["date", "spatial_id", "tci"]].drop_duplicates()
    print(f"  Loaded TCI   : {tci_path.name}  ({len(tci):,} rows, "
          f"{tci['date'].nunique()} dates × {tci['spatial_id'].nunique()} cells)")
    return tci


def add_no2_lag(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a 1-day lagged NO₂ feature (no2_lag1) per (city, spatial_id).

    The lag is computed by sorting on date within each group and shifting by 1.
    Rows where no2_lag1 is NaN (i.e. the first date of each group) are dropped.
    Returns the DataFrame with the new column appended before 'tci' (if present).
    """
    df = df.sort_values(["city", "spatial_id", "date"]).copy()
    df["no2_lag1"] = (
        df.groupby(["city", "spatial_id"])["no2"]
          .shift(1)
    )
    return df


def report_step(label: str, n_before: int, n_after: int, total: int):
    """Print a one-line data-loss report for a processing step."""
    dropped = n_before - n_after
    pct_dropped  = 100 * dropped  / n_before if n_before else 0
    pct_retained = 100 * n_after  / total    if total    else 0
    print(f"  {label:<35} "
          f"dropped {dropped:>8,} rows ({pct_dropped:4.1f}%)  "
          f"→ {n_after:>8,} rows retained ({pct_retained:.1f}% of original)")


print("Helper functions defined.")

## Part 3 — Pre-flight Checks

Verify that both the panel and TCI files exist for each city.

In [ ]:
print("=" * 60)
print("PRE-FLIGHT CHECKS")
print("=" * 60)

all_ok = True

for city in CITIES:
    city_name = city["name"]
    print(f"\n📍 {city['display_name']} ({city_name})")

    # Panel
    panel_dir = DATA_ROOT / "processed" / city_name / "grid_panel"
    panel_found = any(
        (panel_dir / f"{city_name}_grid_panel{ext}").exists()
        for ext in [".csv.gz", ".csv"]
    )
    print(f"  Panel CSV  [{'✓' if panel_found else '✗ MISSING'}]: {panel_dir}")
    if not panel_found:
        all_ok = False

    # Static features (01b5) — optional, soft warning only
    static_path = DATA_ROOT / "processed" / city_name / "grid_panel" / f"{city_name}_static_features.csv"
    static_found = static_path.exists()
    print(f"  Static CSV [{'✓' if static_found else '— optional, not found'}]: {static_path}")
    if not static_found:
        print(f"             (01b5 not yet run for this city — static columns will be NaN)")

    # TCI
    tci_path = DATA_ROOT / "processed" / city_name / "tci" / f"{city_name}_tci_2025.csv"
    tci_found = tci_path.exists()
    print(f"  TCI CSV    [{'✓' if tci_found else '✗ MISSING'}]: {tci_path}")
    if not tci_found:
        all_ok = False

print()
if all_ok:
    print("✅ All hard-blocker checks passed.")
    print("   (Missing static-features files above are non-blocking — see warning per city.)")
else:
    print("⚠️  Some required inputs are missing — check paths above before proceeding.")

## Part 4 — Feature Engineering & TCI Merge

For each city:
1. Load the 01b panel.
2. Merge static spatial features from 01b5 (left join on `spatial_id` — broadcasts across all dates, never drops rows; NaN columns if 01b5 hasn't been run for this city).
3. Optionally add the NO₂ lag feature and drop lag-less rows.
4. Merge TCI (inner join — rows without a TCI match are dropped).
5. Report data loss at each step.

In [ ]:
city_modelling_panels: dict = {}  # city_name → final DataFrame
city_n_original: dict = {}

for city in CITIES:
    city_name    = city["name"]
    display_name = city["display_name"]

    print("\n" + "=" * 60)
    print(f"{display_name} ({city_name})")
    print("=" * 60)

    # --- Load inputs ---------------------------------------------------------
    try:
        panel = load_panel(city_name)
        tci   = load_tci(city_name)
    except FileNotFoundError as e:
        print(f"  ✗ Skipping — {e}")
        continue

    static = load_static_features(city_name)   # None if 01b5 hasn't been run for this city

    n_original = len(panel)
    city_n_original[city_name] = n_original
    print(f"  Original panel: {n_original:,} rows  "
          f"({panel['date'].nunique()} dates × {panel['spatial_id'].nunique()} cells)")

    # --- Step 0: Merge static spatial features (01b5) — left join, never drops rows ----
    # Broadcasts each tile's static value across all dates for that tile.
    n_before_static = len(panel)
    if static is not None:
        static_cols = [c for c in static.columns if c != "spatial_id"]
        panel = panel.merge(static, on="spatial_id", how="left")
        n_missing_tiles = panel[static_cols[0]].isna().sum() if static_cols else 0
        print(f"  Static merge  : {len(static_cols)} columns added  "
              f"({n_missing_tiles:,} rows with no matching tile in 01b5 output, "
              f"{100 * n_missing_tiles / len(panel):.1f}% if any)")
    else:
        # 01b5 output not found — add NaN placeholder columns so the schema stays
        # consistent across cities, rather than silently omitting the columns.
        static_cols = [
            "tile_pop", "building_height", "road_length_total_m",
            "n_bus_stops", "n_mass_transit_lines", "n_mass_transit_polygons",
            "n_public_transport_total", "n_commercial", "n_offices",
            "n_education", "n_leisure", "n_power_plant",
        ]
        for col in static_cols:
            panel[col] = np.nan
        print(f"  Static merge  : skipped (no 01b5 output) — added {len(static_cols)} NaN placeholder columns")
    assert len(panel) == n_before_static, "Static feature merge must never change row count (left join only)"

    # --- spatial_id overlap check --------------------------------------------
    panel_ids = set(panel["spatial_id"].unique())
    tci_ids   = set(tci["spatial_id"].unique())
    overlap   = len(panel_ids & tci_ids)
    print(f"  spatial_id overlap : {overlap:,} / {len(tci_ids):,} TCI cells "
          f"({100 * overlap / len(tci_ids):.1f}%)")

    # --- Step 1: NO₂ lag feature ---------------------------------------------
    if ADD_NO2_LAG:
        if "no2" not in panel.columns:
            print("  ⚠️  ADD_NO2_LAG is True but 'no2' column not found — skipping lag.")
        else:
            panel = panel.sort_values(["city", "spatial_id", "date"]).copy()

            # Shift by 1 row within each (city, spatial_id) group
            panel["no2_lag1"] = panel.groupby(["city", "spatial_id"])["no2"].shift(1)

            # Nullify lag where the previous row is not exactly 1 calendar day prior
            # — prevents cross-interval contamination when date ranges are non-consecutive
            prev_date = panel.groupby(["city", "spatial_id"])["date"].shift(1)
            not_consecutive = (panel["date"] - prev_date) != pd.Timedelta("1D")
            panel.loc[not_consecutive, "no2_lag1"] = np.nan

            n_before_lag_drop = len(panel)
            panel = panel.dropna(subset=["no2_lag1"]).reset_index(drop=True)
            report_step("NO₂ lag — dropped non-consecutive/first-date rows",
                        n_before_lag_drop, len(panel), n_original)
    else:
        print("  [no2_lag1] Skipped (ADD_NO2_LAG = False)")

    n_after_lag = len(panel)

    # --- Step 2: Merge TCI (inner join) --------------------------------------
    panel = panel.merge(
        tci[["date", "spatial_id", "tci"]],
        on=["date", "spatial_id"],
        how="inner",   # drops rows with no matching TCI observation
    )
    report_step("TCI merge — unmatched rows dropped",
                n_after_lag, len(panel), n_original)

    # --- Final sort ----------------------------------------------------------
    panel = panel.sort_values(["spatial_id", "date"]).reset_index(drop=True)
    city_modelling_panels[city_name] = panel

    print(f"  ✓ Final panel : {len(panel):,} rows  "
          f"({panel['date'].nunique()} dates × {panel['spatial_id'].nunique()} cells)")
    print(f"  Columns       : {list(panel.columns)}")

print("\n✅ Feature engineering complete.")

## Part 5 — Missingness Summary

Overview of total data loss across all cities and a per-variable NaN audit of the final panels.

In [ ]:
print("=" * 60)
print("MISSINGNESS SUMMARY")
print("=" * 60)

# --- Overall retention per city ---
print(f"\n{'City':<15} {'Original':>10} {'Final':>10} {'Dropped':>10} {'Retained %':>12}")
print("-" * 60)
for city_name, panel in city_modelling_panels.items():
    city_cfg = next(c for c in CITIES if c["name"] == city_name)
    n_orig   = city_n_original[city_name]
    n_final  = len(panel)
    dropped  = n_orig - n_final
    pct      = 100 * n_final / n_orig
    print(f"{city_cfg['display_name']:<15} {n_orig:>10,} {n_final:>10,} {dropped:>10,} {pct:>11.1f}%")

# --- Per-variable NaN audit ---
print(f"\n{'City':<15} {'Column':<12} {'NaN rows':>10} {'NaN %':>8}")
print("-" * 50)
for city_name, panel in city_modelling_panels.items():
    for col in panel.columns:
        if col in ["city", "spatial_id", "date"]:
            continue
        n_nan = panel[col].isna().sum()
        pct   = 100 * n_nan / len(panel)
        print(f"{city_name:<15} {col:<12} {n_nan:>10,} {pct:>7.1f}%")

## Part 6 — Diagnostic Plots

### 6.1 TCI time series — city-mean daily TCI

In [ ]:
for city_name, panel in city_modelling_panels.items():
    city_cfg   = next((c for c in CITIES if c["name"] == city_name), {})
    daily_mean = panel.groupby("date")["tci"].mean()

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(daily_mean.index, daily_mean.values, linewidth=0.8, color="steelblue")
    ax.set_title(f"{city_cfg.get('display_name', city_name)} — Daily mean TCI", fontsize=12)
    ax.set_ylabel("TCI (mean across cells)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

### 6.2 NO₂ vs NO₂ lag scatter (sample of cells)

In [ ]:
if ADD_NO2_LAG:
    for city_name, panel in city_modelling_panels.items():
        city_cfg = next((c for c in CITIES if c["name"] == city_name), {})

        if "no2_lag1" not in panel.columns:
            continue

        # Sample up to 5000 rows for readability
        sample = panel[["no2", "no2_lag1"]].dropna().sample(
            min(5000, len(panel)), random_state=42
        )

        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(sample["no2_lag1"], sample["no2"],
                   alpha=0.1, s=5, color="steelblue")
        ax.set_xlabel("no2_lag1 (t-1)")
        ax.set_ylabel("no2 (t)")
        ax.set_title(f"{city_cfg.get('display_name', city_name)} — NO₂ vs lag", fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("ADD_NO2_LAG is False — scatter plot skipped.")

## Part 7 — Save Outputs

In [ ]:
for city_name, panel in city_modelling_panels.items():
    out_dir = DATA_ROOT / "processed" / city_name / "grid_panel"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Serialise: ensure date is string, categoricals are strings
    panel_out = panel.copy()
    panel_out["date"] = panel_out["date"].astype(str)
    for col in panel_out.select_dtypes(include="category").columns:
        panel_out[col] = panel_out[col].astype(str)

    # Save as gzip-compressed CSV
    out_path = out_dir / f"{city_name}_modelling_panel.csv.gz"
    panel_out.to_csv(out_path, index=False, compression="gzip")
    size_mb = out_path.stat().st_size / 1e6
    print(f"  {city_name}: {out_path.name}  ({size_mb:.1f} MB, {len(panel):,} rows)")

print("\n✅ All outputs saved.")

## Part 8 — Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 01c SUMMARY")
print("=" * 60)

for city_name, panel in city_modelling_panels.items():
    city_cfg = next((c for c in CITIES if c["name"] == city_name), {})
    out_path = DATA_ROOT / "processed" / city_name / "grid_panel" / f"{city_name}_modelling_panel.csv.gz"
    size     = f"{out_path.stat().st_size / 1e6:.1f} MB" if out_path.exists() else "not found"

    print(f"\n{city_cfg.get('display_name', city_name)}")
    print(f"  Rows         : {len(panel):,}")
    print(f"  Grid cells   : {panel['spatial_id'].nunique():,}")
    print(f"  Dates        : {panel['date'].nunique()}  "
          f"({panel['date'].min()} \u2192 {panel['date'].max()})")
    print(f"  Columns      : {list(panel.columns)}")
    print(f"  NO₂ lag      : {'yes' if 'no2_lag1' in panel.columns else 'no'}")
    print(f"  Output       : {out_path.name}  ({size})")

print("\n\u2192 Next step: 02_panel_assembly.ipynb")